<a href="https://colab.research.google.com/github/imgeG/CYT-300-Group-1/blob/M7/CYT300_M7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup
All files for the mail server are written to `deploy/`. Edit `features.py` only in the main folder; it is copied into `deploy/` here.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil
D      = '/content/drive/MyDrive/CYT300/imge'     # main project folder (features.py, results/)
DEPLOY = f'{D}/deploy'                             # everything VM2 needs, in one folder
os.makedirs(DEPLOY, exist_ok=True)

shutil.copy(f'{D}/features.py', f'{DEPLOY}/features.py')
shutil.copy(f'{D}/results/phish_model_bundle.joblib', f'{DEPLOY}/phish_model_bundle.joblib')
print("Copied features.py and the model bundle to", DEPLOY)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied features.py and the model bundle to /content/drive/MyDrive/CYT300/imge/deploy


## 2. Deployment files

In [2]:
%%writefile /content/drive/MyDrive/CYT300/imge/deploy/scan.py
# scan.py
# Scan one raw email: ML model + strong rules + VirusTotal -> verdict
import time
from email import policy
from email.parser import BytesParser
import joblib
from features import msg_to_row, featurize_one, host_of, first_address, RAW_URL_RE
import vt

class Scanner:
    def __init__(self, bundle_path, max_urls=3):
        b = joblib.load(bundle_path)
        self.model, self.threshold, self.prep = b['model'], b['threshold'], b['prep']
        self.max_urls = max_urls                     # VirusTotal lookups per email (free API is rate-limited)

    @staticmethod
    def _all_urls(msg):
        """Links from EVERY text part (plain and HTML), one per host. Used for VirusTotal only."""
        urls, seen = [], set()
        for part in msg.walk():
            if part.get_content_maintype() != 'text':
                continue
            try:
                text = part.get_content()
            except Exception:
                continue
            for u in RAW_URL_RE.findall(text[:50000]):
                h = host_of(u)
                if h and h not in seen:
                    seen.add(h)
                    urls.append(u)
        return urls

    def scan(self, raw):
        t0 = time.perf_counter()
        msg = BytesParser(policy=policy.default).parsebytes(raw)

        # ML + rules: read the email exactly like training did (From header, plain text preferred, else HTML)
        row = msg_to_row(msg) or {'sender': '', 'receiver': '', 'subject': '', 'body': ''}
        x, fired = featurize_one(row['sender'], row['receiver'], row['subject'], row['body'], self.prep)
        score = None if x is None else float(self.model.predict_proba(x)[0, 1])

        # VirusTotal: reputation of the links
        urls = self._all_urls(msg)
        to_check = urls[:self.max_urls]
        vt_results = vt.check_urls(to_check)
        vt_bad = [u for u, r in vt_results.items() if r['verdict'] == 'malicious']
        checked = sum(r['verdict'] in ('malicious', 'clean', 'unknown') for r in vt_results.values())
        if not urls:
            vt_status = 'no links'
        else:
            vt_status = f"checked {checked}/{len(to_check)}, malicious {len(vt_bad)}"
            if len(urls) > len(to_check):
                vt_status += f", {len(urls) - len(to_check)} more not checked"

        # Final decision: ML OR rules OR VirusTotal
        reasons = []
        if not first_address(row['sender']):          # no valid From address: never seen in legit mail
            reasons.append("rule:no_valid_sender")
        if score is not None and score >= self.threshold:
            reasons.append(f"ml:{score:.3f}")
        reasons += [f"rule:{r}" for r in (fired or [])]
        reasons += [f"vt:{host_of(u)}" for u in vt_bad]

        if reasons:
            verdict = 'PHISHING'
        elif score is None:
            verdict = 'UNSCORED'                     # no From address or no text: ML could not run
        else:
            verdict = 'CLEAN'

        return {'verdict': verdict, 'score': score, 'threshold': self.threshold,
                'rules': fired or [], 'reasons': reasons, 'urls_found': len(urls),
                'vt': vt_results, 'vt_status': vt_status,
                'vt_api_calls': sum(r.get('source') == 'api' for r in vt_results.values()),
                'ms': round((time.perf_counter() - t0) * 1000, 1)}

Overwriting /content/drive/MyDrive/CYT300/imge/deploy/scan.py


In [3]:
%%writefile /content/drive/MyDrive/CYT300/imge/deploy/vt.py
# vt.py
# VirusTotal URL reputation: lookup only, cached, rate-limited. Never holds up mail if VT is slow or down
import base64, os, sqlite3, threading, time
import requests

API_KEY = os.environ.get('VT_API_KEY', '').strip()
if not API_KEY:
    raise RuntimeError("VT_API_KEY is not set. VirusTotal is required: put the key in "
                       "/etc/phish/phish.env (VM2) or Colab Secrets (Colab).")

CACHE_PATH    = os.environ.get('VT_CACHE', 'vt_cache.sqlite')
PER_MINUTE    = int(os.environ.get('VT_PER_MINUTE', '4'))        # free API quota; check your account page
MIN_MALICIOUS = int(os.environ.get('VT_MIN_MALICIOUS', '1'))     # engines that must say "malicious"
TTL_KNOWN, TTL_UNKNOWN, TIMEOUT = 24 * 3600, 3600, 5

_lock, _calls = threading.Lock(), []
_db = sqlite3.connect(CACHE_PATH, check_same_thread=False)
_db.execute('CREATE TABLE IF NOT EXISTS vt (url TEXT PRIMARY KEY, verdict TEXT, malicious INT, checked REAL)')

def _url_id(url):
    return base64.urlsafe_b64encode(url.encode()).decode().rstrip('=')

def _from_cache(url):
    with _lock:
        row = _db.execute('SELECT verdict, malicious, checked FROM vt WHERE url = ?', (url,)).fetchone()
    if row:
        ttl = TTL_UNKNOWN if row[0] == 'unknown' else TTL_KNOWN
        if time.time() - row[2] < ttl:
            return {'verdict': row[0], 'malicious': row[1], 'source': 'cache'}
    return None

def _save(url, verdict, malicious):
    with _lock:
        _db.execute('INSERT OR REPLACE INTO vt VALUES (?, ?, ?, ?)', (url, verdict, malicious, time.time()))
        _db.commit()

def _quota_ok():
    with _lock:
        now = time.time()
        _calls[:] = [t for t in _calls if now - t < 60]
        if len(_calls) >= PER_MINUTE:
            return False
        _calls.append(now)
        return True

def check_url(url):
    cached = _from_cache(url)
    if cached:
        return cached
    if not _quota_ok():
        return {'verdict': 'skipped_rate_limit'}
    try:
        r = requests.get('https://www.virustotal.com/api/v3/urls/' + _url_id(url),
                         headers={'x-apikey': API_KEY}, timeout=TIMEOUT)
    except requests.RequestException:
        return {'verdict': 'error'}
    if r.status_code == 404:                               # VirusTotal has never seen this URL
        _save(url, 'unknown', 0)
        return {'verdict': 'unknown', 'malicious': 0, 'source': 'api'}
    if r.status_code != 200:                               # 429 = quota used up, others = API problem
        return {'verdict': 'skipped_rate_limit' if r.status_code == 429 else 'error'}
    stats = r.json().get('data', {}).get('attributes', {}).get('last_analysis_stats', {})
    mal = int(stats.get('malicious', 0))
    verdict = 'malicious' if mal >= MIN_MALICIOUS else 'clean'
    _save(url, verdict, mal)
    return {'verdict': verdict, 'malicious': mal, 'source': 'api'}

def check_urls(urls):
    return {u: check_url(u) for u in urls}

def self_test():
    """Confirm the key works (one lookup). Raises if VirusTotal rejects the key."""
    r = requests.get('https://www.virustotal.com/api/v3/urls/' + _url_id('https://www.google.com/'),
                     headers={'x-apikey': API_KEY}, timeout=TIMEOUT)
    if r.status_code == 401:
        raise RuntimeError("VirusTotal rejected the API key (HTTP 401). Check VT_API_KEY.")
    return r.status_code

Overwriting /content/drive/MyDrive/CYT300/imge/deploy/vt.py


In [4]:
%%writefile /content/drive/MyDrive/CYT300/imge/deploy/phish_milter.py
# phish_milter.py
# Postfix milter: scans each incoming email and adds X-Phish-* headers (flag only, never blocks)
import logging, os, sys
from io import BytesIO
import Milter, requests
import vt
from scan import Scanner

BUNDLE      = os.environ.get('PHISH_BUNDLE', '/opt/phish/phish_model_bundle.joblib')
SOCKET      = os.environ.get('PHISH_SOCKET', 'inet:8899@127.0.0.1')
TAG_SUBJECT = os.environ.get('PHISH_TAG_SUBJECT', '1') == '1'   # prefix the subject (visible in Thunderbird)
MAX_BYTES   = 5 * 1024 * 1024                                    # stop buffering huge emails
OUR_HEADERS = ('x-phish-verdict', 'x-phish-score', 'x-phish-reasons', 'x-phish-vt')

logging.basicConfig(stream=sys.stdout, level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('phishmilter')
SCANNER = Scanner(BUNDLE)
log.info("model loaded, threshold %.4f", SCANNER.threshold)
try:
    log.info("VirusTotal key check: HTTP %s", vt.self_test())   # wrong key -> raises -> service stops
except requests.RequestException as e:
    log.warning("VirusTotal not reachable at startup (%s); will retry on each email", e)

class PhishMilter(Milter.Base):
    def __init__(self):
        self.id = Milter.uniqueID()

    def envfrom(self, mailfrom, *args):                # start of each new email on the connection
        self.buf, self.size, self.subject, self.forged = BytesIO(), 0, None, {}
        return Milter.CONTINUE

    def _keep(self, data):
        if self.size < MAX_BYTES:
            self.buf.write(data)
            self.size += len(data)

    def header(self, name, value):
        lname = name.lower()
        if lname in OUR_HEADERS:                       # sender tried to pre-set our verdict -> remove later
            self.forged[name] = self.forged.get(name, 0) + 1
        if lname == 'subject' and self.subject is None:
            self.subject = value
        self._keep(f"{name}: {value}\r\n".encode('utf-8', 'surrogateescape'))
        return Milter.CONTINUE

    def eoh(self):
        self._keep(b"\r\n")
        return Milter.CONTINUE

    def body(self, chunk):
        self._keep(chunk)
        return Milter.CONTINUE

    def eom(self):
        qid = self.getsymval('i') or self.id
        try:
            res = SCANNER.scan(self.buf.getvalue())
        except Exception:
            log.exception("%s scan failed, accepting unchanged", qid)
            return Milter.ACCEPT                        # fail-open: never lose mail because of the scanner

        for name, count in self.forged.items():         # strip forged verdict headers
            for i in range(count, 0, -1):
                self.chgheader(name, i, None)

        score = '-' if res['score'] is None else f"{res['score']:.3f}"
        self.addheader('X-Phish-Verdict', res['verdict'])
        self.addheader('X-Phish-Score', f"{score} (threshold {res['threshold']:.3f})")
        self.addheader('X-Phish-Reasons', ', '.join(res['reasons']) or 'none')
        self.addheader('X-Phish-VT', res['vt_status'])

        if TAG_SUBJECT and res['verdict'] == 'PHISHING':
            new = '[SUSPECTED PHISHING] ' + (self.subject or '')
            if self.subject is None:
                self.addheader('Subject', new)
            else:
                self.chgheader('Subject', 1, new)

        log.info("%s verdict=%s score=%s reasons=%s vt=[%s] ms=%.1f",
                 qid, res['verdict'], score, res['reasons'], res['vt_status'], res['ms'])
        return Milter.ACCEPT

if __name__ == '__main__':
    Milter.factory = PhishMilter
    Milter.set_flags(Milter.ADDHDRS | Milter.CHGHDRS)  # we add headers and change Subject
    log.info("listening on %s", SOCKET)
    Milter.runmilter('phishmilter', SOCKET, timeout=600)

Overwriting /content/drive/MyDrive/CYT300/imge/deploy/phish_milter.py


In [5]:
%%writefile /content/drive/MyDrive/CYT300/imge/deploy/phishmilter.service
[Unit]
Description=Phishing detection milter (ML + rules + VirusTotal)
After=network-online.target
Wants=network-online.target
Before=postfix.service

[Service]
User=phish
Group=phish
WorkingDirectory=/opt/phish
EnvironmentFile=/etc/phish/phish.env
Environment=VT_CACHE=/var/lib/phish/vt_cache.sqlite
StateDirectory=phish
ExecStart=/opt/phish/venv/bin/python /opt/phish/phish_milter.py
Restart=on-failure
RestartSec=5
NoNewPrivileges=yes
ProtectSystem=strict
ProtectHome=yes
PrivateTmp=yes

[Install]
WantedBy=multi-user.target

Overwriting /content/drive/MyDrive/CYT300/imge/deploy/phishmilter.service


In [6]:
# syntax check
# The milter can't run in Colab (no Postfix), but we can check every file compiles
import py_compile
for f in ['features.py', 'scan.py', 'vt.py', 'phish_milter.py']:
    py_compile.compile(f'{DEPLOY}/{f}', doraise=True)
    print("OK", f)
print(sorted(os.listdir(DEPLOY)))

OK features.py
OK scan.py
OK vt.py
OK phish_milter.py
['__pycache__', 'features.py', 'phish_milter.py', 'phish_model_bundle.joblib', 'phishmilter.service', 'scan.py', 'vt.py']


## 3. Tests in Colab
Needs the VirusTotal key in Colab Secrets (key icon on the left, name `VT_API_KEY`, notebook access on).

In [7]:
# load the scanner and check the key
from google.colab import userdata
os.environ['VT_API_KEY'] = userdata.get('VT_API_KEY')     # must be set BEFORE importing vt
os.environ['VT_CACHE']   = '/content/vt_cache.sqlite'

sys.path.insert(0, DEPLOY)
import importlib, features, vt, scan
for m in (features, vt, scan):
    importlib.reload(m)

print("VirusTotal key check: HTTP", vt.self_test())        # expect 200
S = scan.Scanner(f'{DEPLOY}/phish_model_bundle.joblib')
print("Scanner ready | threshold", round(S.threshold, 4))

VirusTotal key check: HTTP 200
Scanner ready | threshold 0.9109


In [8]:
# two sample emails
from email.message import EmailMessage

def make_email(sender, to, subject, body, html=False):
    m = EmailMessage()
    m['From'], m['To'], m['Subject'] = sender, to, subject
    m.set_content(body, subtype='html' if html else 'plain')
    return m.as_bytes()

samples = {
    'phishing': make_email('PayPal Support <support@secure-mail.xyz>', 'user@example.com',
                           'Your account is suspended',
                           'Please verify your account immediately: http://192.168.1.10/login'),
    'legit':    make_email('Sarah Lee <sarah.lee@company.com>', 'user@company.com', 'Re: meeting notes',
                           'Hi, attached are the notes from Tuesday. Let me know if I missed anything. Thanks, Sarah'),
}
for name, raw in samples.items():
    r = S.scan(raw)
    print(f"{name:9s} -> {r['verdict']:8s} | score {r['score']:.3f} | reasons {r['reasons']} | "
          f"VT: {r['vt_status']} | {r['ms']} ms")

phishing  -> PHISHING | score 1.000 | reasons ['ml:1.000', 'rule:s_display_brand_mismatch', 'rule:u_ip'] | VT: checked 1/1, malicious 0 | 124.1 ms
legit     -> CLEAN    | score 0.002 | reasons [] | VT: no links | 79.7 ms


In [9]:
# 300 emails from the test set, with timing
import pandas as pd, numpy as np

def to_raw(r):
    m = EmailMessage()
    for h, k in (('From', 'sender'), ('To', 'receiver'), ('Subject', 'subject')):
        v = ' '.join(str(r[k]).split())                  # no line breaks inside headers
        if v:
            try:
                m[h] = v
            except Exception:
                pass
    body = str(r['body'])
    m.set_content(body, subtype='html' if '<html' in body.lower() else 'plain')
    return m.as_bytes()

t = pd.read_parquet(f'{D}/cache/test_raw.parquet').sample(300, random_state=42)
rows = []
for _, r in t.iterrows():
    try:
        res = S.scan(to_raw(r))
        rows.append({'label': r['label'], 'verdict': res['verdict'], 'ms': res['ms'],
                     'vt_api_calls': res['vt_api_calls'], 'vt_status': res['vt_status']})
    except Exception as e:
        rows.append({'label': r['label'], 'verdict': 'ERROR', 'ms': np.nan, 'vt_api_calls': 0, 'vt_status': str(e)[:60]})
out = pd.DataFrame(rows)

print(pd.crosstab(out['label'].map({0: 'legit', 1: 'phishing'}), out['verdict']), "\n")
no_api, with_api = out[out['vt_api_calls'] == 0]['ms'], out[out['vt_api_calls'] > 0]['ms']
print(f"time per email WITHOUT a VirusTotal call: median {no_api.median():.1f} ms | 95th pct {no_api.quantile(.95):.1f} ms")
if len(with_api):
    print(f"time per email WITH a VirusTotal call:    median {with_api.median():.1f} ms ({len(with_api)} emails)")
print("emails where VirusTotal was skipped (rate limit):", out['vt_status'].str.contains('checked 0').sum())
out.to_csv(f'{D}/results/m7_colab_scan_test.csv', index=False)

verdict   CLEAN  ERROR  PHISHING
label                           
legit       127      0        11
phishing     24      1       137 

time per email WITHOUT a VirusTotal call: median 32.4 ms | 95th pct 138.6 ms
time per email WITH a VirusTotal call:    median 465.6 ms (4 emails)
emails where VirusTotal was skipped (rate limit): 102


In [10]:
# Senders with no valid address: legit vs phishing
tt = t.reset_index(drop=True)
raw = pd.read_parquet(f'{D}/cache/test_raw.parquet')
has_addr = raw['sender'].fillna('').astype(str).apply(features.first_address) != ''
print(raw.loc[~has_addr, 'sender'].head(20).tolist())        # examples of senders with no valid address

# How often is the sender address missing in the FULL raw test set, legit vs phishing?
print(pd.crosstab(raw['label'].map({0: 'legit', 1: 'phishing'}), has_addr.rename('has sender address')))

['Selektiert!', '"The Sapphire Studio", <>', 'Netflix <info@{members.netflix.com}>', '"your bonus phishing"@pot, <>', '"Casino rewards phishing"@pot, <>', 'Correios <notifica@correios>', 'Costco Membership Center <XJLWDQVA@Z4KOYGL0YL4F6W2YOBZGZ5V8YOJI>', 'Account security <1ZB79D1Y@Z35RLBQUI8RKM9H1PWKRJEENBD06>', '💼 Ihre BTC Auszahlung <newsletter@0mw89>', 'Shein Opinion Requested <shein-survey-free>, net, <>', 'Play Now <>, <>', 'Correios <noreply@correios>', '"Bank check phishing"@pot, <>', 'Correios <notificacoes@correios>', '"Casino rewards phishing"@pot, <>', '🔐 Cloud Support <HIAJ0JM2@FHQ18KPAXU3VZUAHL3CR7PVXOJC7>', 'Shein Opinion Requested <shein-survey-free>, net, <>', '"Casino rewards phishing"@pot, <>', 'Willkommensbonus', 'Correios <noreply@correios>']
has sender address  False  True 
label                           
legit                   0   3000
phishing              235   2765


In [13]:
# Measure what VirusTotal adds
import time
vt._db.execute('DELETE FROM vt'); vt._db.commit()     # empty the cache so every lookup is a real VirusTotal call
print("VirusTotal cache cleared")
S1 = scan.Scanner(f'{DEPLOY}/phish_model_bundle.joblib', max_urls=1)   # 1 lookup per email to save quota
time.sleep(60)                                                          # let the per-minute quota reset

with_links = tt[tt['body'].astype(str).str.contains('http', case=False)]
pick = pd.concat([with_links[with_links['label'] == 1].head(15), with_links[with_links['label'] == 0].head(5)])
vt_rows = []
for _, r in pick.iterrows():
    try:
        res = S1.scan(to_raw(r))
        vt_rows.append({'label': 'phishing' if r['label'] else 'legit', 'verdict': res['verdict'],
                        'vt': res['vt_status'], 'vt_flagged': any(x.startswith('vt:') for x in res['reasons']),
                        'ml_or_rules_flagged': any(not x.startswith('vt:') for x in res['reasons']), 'ms': res['ms']})
    except Exception as e:
        vt_rows.append({'label': r['label'], 'verdict': 'ERROR', 'vt': str(e)[:50]})
    time.sleep(16)                                                      # stay under 4 lookups/minute
vt_out = pd.DataFrame(vt_rows)
print(vt_out.to_string(index=False))
print("\nVirusTotal flagged:", vt_out.groupby('label')['vt_flagged'].sum().to_dict())
vt_out.to_csv(f'{D}/results/m7_vt_test.csv', index=False)

VirusTotal cache cleared
   label  verdict                                                 vt vt_flagged ml_or_rules_flagged     ms
phishing PHISHING       checked 1/1, malicious 0, 2 more not checked      False                True 1044.2
phishing PHISHING                                           no links      False                True  134.4
phishing    CLEAN                                           no links      False               False  118.5
phishing PHISHING                                           no links      False                True  118.5
phishing PHISHING       checked 1/1, malicious 0, 2 more not checked      False                True  482.5
phishing PHISHING       checked 1/1, malicious 0, 7 more not checked      False                True  466.9
       1    ERROR folded header contains newline: b'From: =?utf-8?b?        NaN                 NaN    NaN
phishing PHISHING                                           no links      False                True  206.7
phishing    

In [12]:
vt_out['label'] = vt_out['label'].replace({1: 'phishing', 0: 'legit'})
ok = vt_out['verdict'] != 'ERROR'
v = vt_out[ok]
only_vt = v['vt_flagged'].astype(bool) & ~v['ml_or_rules_flagged'].astype(bool)
print("Phishing caught by ML+rules:", int(v[v.label == 'phishing']['ml_or_rules_flagged'].sum()),
      "| with VirusTotal added:", int((v[v.label == 'phishing'][['vt_flagged', 'ml_or_rules_flagged']].any(axis=1)).sum()),
      "| out of", int((v.label == 'phishing').sum()))
print("Caught ONLY by VirusTotal:", int(only_vt.sum()))
print("Legit flagged by VirusTotal:", int(v[v.label == 'legit']['vt_flagged'].sum()))
vt_out.to_csv(f'{D}/results/m7_vt_test.csv', index=False)

Phishing caught by ML+rules: 11 | with VirusTotal added: 12 | out of 14
Caught ONLY by VirusTotal: 1
Legit flagged by VirusTotal: 0


## 4. Deploy on VM2
1. Download the `deploy` folder from Google Drive to your laptop, then copy it to VM2 over Tailscale:
   `scp -r deploy USER@VM2:/tmp/` then on VM2: `sudo mkdir -p /opt/phish && sudo cp /tmp/deploy/* /opt/phish/`
2. Install: `sudo apt install -y python3-venv python3-dev build-essential libmilter-dev swaks`
   `sudo python3 -m venv /opt/phish/venv`
   `sudo /opt/phish/venv/bin/pip install scikit-learn==1.6.1 numpy==2.1.3 pandas==2.2.3 scipy joblib pymilter requests`
3. Service user + key: `sudo useradd --system --no-create-home --shell /usr/sbin/nologin phish`
   `sudo mkdir -p /etc/phish && sudo nano /etc/phish/phish.env`  (one line: `VT_API_KEY=...`)
   `sudo chmod 600 /etc/phish/phish.env`
4. Service: `sudo cp /opt/phish/phishmilter.service /etc/systemd/system/`
   `sudo systemctl daemon-reload && sudo systemctl enable --now phishmilter`
   `journalctl -u phishmilter -f`  (expect: model loaded, VirusTotal key check: HTTP 200, listening)
5. Postfix: add to /etc/postfix/main.cf:
   `smtpd_milters = inet:127.0.0.1:8899`,
   `milter_default_action = accept`, `milter_protocol = 6`
   Check master.cf `submission` has no `-o smtpd_milters=` override. Then `sudo postfix check && sudo systemctl reload postfix`
6. Test with swaks from the attacker account, check Thunderbird subject + View > Message Source, and the journal.